# Machine Learning with Scikit-Learn



---

## Learning Objectives

- Build KNN and Decision Tree classifiers using Scikit-learn.
- Use the Pipeline API to chain preprocessing + model.
- Apply K-Means clustering with Elbow and Silhouette analysis.
- Train an MLPClassifier (neural network) for digit recognition.
- Compare multiple models on the same dataset.

## Concept / Theory

### KNN (K-Nearest Neighbors)
Classifies new points by majority vote of their K closest training neighbors. Simple but powerful for small datasets.

### Decision Tree
Splits data at each node using the best feature+threshold to minimize impurity (Gini or Entropy). Interpretable — you can visualize it.

### Sklearn Pipeline
Chains steps (scaler, feature selector, model) into one object. Prevents data leakage and simplifies cross-validation.

### K-Means Clustering
Unsupervised algorithm that partitions data into K clusters by minimizing within-cluster distance. Useful for customer segmentation, data compression.

### MLPClassifier
Multi-Layer Perceptron in Scikit-learn. A feed-forward neural network for tabular data.

In [ ]:
# Setup
!pip install -q scikit-learn matplotlib seaborn pandas numpy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.cluster import KMeans
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (accuracy_score, classification_report,
                              confusion_matrix, ConfusionMatrixDisplay, silhouette_score)

# Load datasets
iris   = datasets.load_iris()
digits = datasets.load_digits()
print('Libraries loaded!')
print(f'Iris: {iris.data.shape} | Digits: {digits.data.shape}')

## Practical Lab 1 – K-Nearest Neighbors Classifier

### Objective
Train and evaluate a KNN classifier. Explore the effect of K.

### Explanation
KNN stores all training samples and classifies new points by finding the K closest neighbors (by Euclidean distance) and taking a majority vote.

In [ ]:
# Lab 1: KNN on Iris dataset
X, y = iris.data, iris.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

scaler = StandardScaler()
X_tr = scaler.fit_transform(X_train)
X_te = scaler.transform(X_test)

# Try different K values
print('K | Accuracy')
print('-----------')
for k in [1, 3, 5, 7, 9, 11]:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_tr, y_train)
    acc = accuracy_score(y_test, knn.predict(X_te))
    print(f'{k} | {acc:.4f}')

In [ ]:
# Best KNN model evaluation
best_knn = KNeighborsClassifier(n_neighbors=5)
best_knn.fit(X_tr, y_train)
y_pred = best_knn.predict(X_te)

print('KNN (k=5) Accuracy:', accuracy_score(y_test, y_pred))
print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=iris.target_names))

### Expected Output
```
5 | 0.9667
KNN (k=5) Accuracy: 0.9667
```

## Practical Lab 2 – Decision Tree Classifier & Visualization

### Objective
Train a Decision Tree and visualize its decision rules.

### Explanation
Decision Trees are the most interpretable ML model. You can literally see every decision the model makes. `max_depth` controls complexity — deeper trees risk overfitting.

In [ ]:
# Lab 2: Decision Tree
dt = DecisionTreeClassifier(max_depth=3, random_state=42)
dt.fit(X_tr, y_train)
y_pred_dt = dt.predict(X_te)

print('Decision Tree Accuracy:', accuracy_score(y_test, y_pred_dt))
print(classification_report(y_test, y_pred_dt, target_names=iris.target_names))

In [ ]:
# Visualize the tree
plt.figure(figsize=(14, 6))
plot_tree(dt, feature_names=iris.feature_names, class_names=iris.target_names,
          filled=True, rounded=True, fontsize=9)
plt.title('Decision Tree (max_depth=3) — Iris Dataset')
plt.tight_layout()
plt.show()

# Feature importances
print('\nFeature Importances:')
for name, imp in zip(iris.feature_names, dt.feature_importances_):
    bar = '█' * int(imp * 40)
    print(f'  {name:<25} {imp:.3f} {bar}')

## Practical Lab 3 – Sklearn Pipeline API

### Objective
Build clean ML pipelines that prevent data leakage.

### Explanation
The Pipeline chains steps sequentially. When you call `pipeline.fit(X_train, y_train)`, it fits each step in order. When you call `predict()`, each step transforms first.

In [ ]:
# Lab 3: Pipeline — clean & reusable
knn_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', KNeighborsClassifier(n_neighbors=5))
])

dt_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', DecisionTreeClassifier(max_depth=3, random_state=42))
])

# Fit on raw (unscaled) training data — pipeline handles scaling internally
knn_pipe.fit(X_train, y_train)
dt_pipe.fit(X_train, y_train)

print('Pipeline KNN accuracy:', accuracy_score(y_test, knn_pipe.predict(X_test)))
print('Pipeline DT  accuracy:', accuracy_score(y_test, dt_pipe.predict(X_test)))
print('\nPipeline steps:', [step[0] for step in knn_pipe.steps])

In [ ]:
# Compare on Breast Cancer dataset
from sklearn.datasets import load_breast_cancer
cancer = load_breast_cancer()
Xc, yc = cancer.data, cancer.target
Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(Xc, yc, test_size=0.2, stratify=yc, random_state=42)

models = {
    'KNN-5':          Pipeline([('sc', StandardScaler()), ('clf', KNeighborsClassifier(5))]),
    'DT depth=5':     Pipeline([('sc', StandardScaler()), ('clf', DecisionTreeClassifier(5, random_state=42))]),
    'LogReg':         Pipeline([('sc', StandardScaler()), ('clf', __import__('sklearn.linear_model', fromlist=['LogisticRegression']).LogisticRegression(max_iter=500, random_state=42))]),
}

print('Model Comparison — Breast Cancer Dataset')
print(f'{"Model":<15} {"Accuracy":>10}')
print('-' * 28)
for name, pipe in models.items():
    pipe.fit(Xc_tr, yc_tr)
    acc = accuracy_score(yc_te, pipe.predict(Xc_te))
    print(f'{name:<15} {acc:>10.4f}')

## Practical Lab 4 – K-Means Clustering

### Objective
Cluster synthetic customer data and choose the optimal K.

### Explanation
K-Means assigns each point to the nearest cluster center, then updates centers. The **Elbow Method** looks for the K where adding more clusters gives diminishing returns (inertia). **Silhouette Score** measures cluster separation quality (closer to 1 is better).

In [ ]:
# Lab 4: Create synthetic customer data
np.random.seed(42)
group1 = np.random.randn(50, 2) * 1.0 + [2, 2]
group2 = np.random.randn(50, 2) * 1.0 + [8, 3]
group3 = np.random.randn(50, 2) * 1.0 + [5, 8]

X_cust = np.vstack([group1, group2, group3])
df_cust = pd.DataFrame(X_cust, columns=['Income', 'Spending'])
print('Customer data shape:', X_cust.shape)

plt.figure(figsize=(6, 5))
plt.scatter(X_cust[:,0], X_cust[:,1], alpha=0.6)
plt.xlabel('Income'); plt.ylabel('Spending')
plt.title('Customer Data (unlabeled)')
plt.show()

In [ ]:
# Elbow method
X_sc = StandardScaler().fit_transform(X_cust)
inertias  = []
silhouettes = []
K_range = range(2, 9)
for k in K_range:
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    labels = km.fit_predict(X_sc)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_sc, labels))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(K_range, inertias, 'bo-')
axes[0].set_xlabel('K'); axes[0].set_ylabel('Inertia'); axes[0].set_title('Elbow Method')
axes[1].plot(K_range, silhouettes, 'ro-')
axes[1].set_xlabel('K'); axes[1].set_ylabel('Silhouette Score'); axes[1].set_title('Silhouette Analysis')
plt.tight_layout(); plt.show()
print('Best silhouette at K=', K_range[silhouettes.index(max(silhouettes))])

In [ ]:
# Apply K-Means with optimal K=3
km = KMeans(n_clusters=3, n_init=10, random_state=42)
labels = km.fit_predict(X_sc)
df_cust['Cluster'] = labels

plt.figure(figsize=(7, 5))
for k in range(3):
    mask = labels == k
    plt.scatter(X_cust[mask, 0], X_cust[mask, 1], label=f'Cluster {k}', alpha=0.7)
centers = km.cluster_centers_
plt.scatter(centers[:,0]*X_cust.std()+X_cust.mean(), centers[:,1]*X_cust.std()+X_cust.mean(),
            c='black', marker='X', s=200, label='Centroids')
plt.xlabel('Income'); plt.ylabel('Spending')
plt.title('K-Means Clustering (K=3)')
plt.legend(); plt.show()

print('Cluster summary:')
print(df_cust.groupby('Cluster')[['Income','Spending']].mean().round(2))

##  Practical Lab 5 – MLPClassifier on Digits Dataset

### Objective
Train a Multi-Layer Perceptron to recognize handwritten digits.

### Explanation
The Digits dataset contains 8×8 grayscale images of handwritten digits (0-9). MLPClassifier implements a feed-forward neural network using backpropagation.

In [ ]:
# Lab 5: Digits dataset
X_d, y_d = digits.data, digits.target
print(f'Digits dataset: {X_d.shape} samples, {len(np.unique(y_d))} classes')

# Visualize some samples
fig, axes = plt.subplots(2, 10, figsize=(15, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(digits.images[i], cmap='gray_r')
    ax.set_title(y_d[i])
    ax.axis('off')
plt.suptitle('Sample digits (0-9)', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Train MLPClassifier
Xd_tr, Xd_te, yd_tr, yd_te = train_test_split(X_d, y_d, test_size=0.2, stratify=y_d, random_state=42)

mlp_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('mlp', MLPClassifier(hidden_layer_sizes=(128, 64),
                          activation='relu',
                          max_iter=200,
                          random_state=42,
                          verbose=False))
])
mlp_pipe.fit(Xd_tr, yd_tr)
y_pred_mlp = mlp_pipe.predict(Xd_te)

print('MLP Accuracy on Digits:', accuracy_score(yd_te, y_pred_mlp))
print('\nClassification Report:')
print(classification_report(yd_te, y_pred_mlp))

In [ ]:
# Confusion matrix for digits
cm = confusion_matrix(yd_te, y_pred_mlp)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted'); plt.ylabel('Actual')
plt.title('Confusion Matrix — MLP on Digits')
plt.tight_layout()
plt.show()

### Expected Output
```
MLP Accuracy on Digits: ~0.978
```

##  Exercises

**Exercise 1:** On the Iris dataset, try KNN with k=1, 5, 15. Which performs best on the test set?

In [ ]:
# Exercise 1
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
iris = load_iris()
X, y = iris.data, iris.target
# Try k=1, 5, 15 and compare

**Exercise 2:** Train a Decision Tree on Digits dataset with `max_depth` 3, 5, and 10. Compare accuracies.

In [ ]:
# Exercise 2
# Use digits dataset and test different max_depth values for Decision Tree

**Exercise 3:** Use the Elbow method on the Iris dataset (X only, ignore y). Which K does the elbow suggest?

In [ ]:
# Exercise 3
from sklearn.datasets import load_iris
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
iris = load_iris()
X = StandardScaler().fit_transform(iris.data)
# Plot elbow curve

## 10. Challenge Task

Use the Breast Cancer dataset. Train ALL four models (KNN, DT, MLP, LogReg) in pipelines. Build a comparison table showing accuracy, training time, and prediction time for each.

In [ ]:
# Challenge Task: Model Comparison
import time
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import pandas as pd

cancer = load_breast_cancer()
X, y = cancer.data, cancer.target
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

pipelines = {
    'LogisticRegression': Pipeline([('sc', StandardScaler()), ('clf', LogisticRegression(max_iter=500, random_state=42))]),
    'KNN (k=5)':          Pipeline([('sc', StandardScaler()), ('clf', KNeighborsClassifier(5))]),
    'Decision Tree':      Pipeline([('sc', StandardScaler()), ('clf', DecisionTreeClassifier(max_depth=5, random_state=42))]),
    'MLP':                Pipeline([('sc', StandardScaler()), ('clf', MLPClassifier(hidden_layer_sizes=(64,32), max_iter=200, random_state=42))]),
}

results = []
for name, pipe in pipelines.items():
    t0 = time.time()
    pipe.fit(X_tr, y_tr)
    t_train = time.time() - t0
    t0 = time.time()
    acc = accuracy_score(y_te, pipe.predict(X_te))
    t_pred = time.time() - t0
    results.append({'Model': name, 'Accuracy': round(acc, 4), 'Train(s)': round(t_train, 3), 'Pred(ms)': round(t_pred*1000, 2)})

results_df = pd.DataFrame(results).sort_values('Accuracy', ascending=False).reset_index(drop=True)
print(results_df.to_string(index=False))

## 11. Real-World AI/ML Application

| Algorithm | Real Use Case |
|---|---|
| **KNN** | Recommendation systems, anomaly detection |
| **Decision Tree** | Credit risk scoring, medical diagnosis rules |
| **K-Means** | Customer segmentation, image compression, document clustering |
| **MLP** | Fraud detection, tabular data classification |
| **Pipeline** | Production ML systems where data leakage must be prevented |

## 12. Day Summary

| Model | Type | Key Parameter |
|---|---|---|
| KNN | Supervised classification | `n_neighbors=K` |
| Decision Tree | Supervised classification | `max_depth` |
| K-Means | Unsupervised clustering | `n_clusters=K` |
| MLPClassifier | Supervised NN | `hidden_layer_sizes` |

**Pipeline:** Always use `Pipeline([('scaler', ...), ('model', ...)])` in production code.

**→ Day 5: Supervised Learning — Classification & Regression**